# Simple Personal Knowledge Worker (RAG) – Local Ollama

This notebook demonstrates a **very simple Retrieval Augmented Generation (RAG)** system using a local LLM.

### Tools
- Ollama (local LLM)
- Llama 3.2 model
- Simple vector retrieval

### Steps
1. Create some example personal knowledge documents
2. Convert them into embeddings
3. Store them in a vector database
4. Retrieve relevant information when a question is asked
5. Use the LLM to generate an answer using the retrieved context

This simulates a **personal knowledge assistant** that can answer questions about stored information.

In [ ]:
!pip install langchain langchain-community langchain-text-splitters chromadb ollama

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama

# -----------------------------------
# 1 Create example personal documents
# -----------------------------------

documents = [
    Document(page_content="Silas is a software developer with experience in Python, JavaScript, and machine learning."),
    Document(page_content="Silas teaches programming and enjoys helping students learn algorithms and coding."),
    Document(page_content="Silas is currently learning about AI engineering, RAG systems, and local LLMs."),
    Document(page_content="Silas is interested in Android development using Kotlin and Android Studio.")
]

# -----------------------------------
# 2 Split documents into chunks
# -----------------------------------

splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.split_documents(documents)

# -----------------------------------
# 3 Create embeddings using Ollama
# -----------------------------------

embeddings = OllamaEmbeddings(model="llama3.2")

# -----------------------------------
# 4 Create vector database
# -----------------------------------

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={"k":2})

# -----------------------------------
# 5 Load local LLM
# -----------------------------------

llm = Ollama(model="llama3.2")

print("RAG system ready.")

In [ ]:
# Ask a question
question = "What is Silas currently learning?"

# Retrieve relevant documents
docs = retriever.invoke(question)

context = "\n".join([d.page_content for d in docs])

# Build prompt
prompt = f"""
Use the context below to answer the question.

Context:
{context}

Question:
{question}
"""

# Generate response
response = llm.invoke(prompt)

print("Question:", question)
print("\nAnswer:")
print(response)